In [35]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [36]:
# SpContext.stop() # Remove the comment if you want to re-run this cell
conf = SparkConf().setMaster("local").setAppName("RegressionAssignment") # treat every core of your desktop as an executor
# creating a spark context object named "SpContext"
SpContext = SparkContext(conf = conf)

In [37]:
# Create a SparkSession object named "SpSession" (the config bit is only for Windows!)
# Initialize a SparkSession with the SQL warehouse directory set
SpSession = SparkSession.builder \
    .appName("RegressionAssignment") \
    .config("spark.sql.warehouse.dir", "/content") \
    .getOrCreate()

In [38]:
#Load the CSV file into a RDD
"""--------------------------------------------------------------------------
Load Data
-------------------------------------------------------------------------"""
# Use the raw URL to download the CSV content, not the HTML page
!rm -f 7-cars-miles-per-gallon.csv
!wget -O 7-cars-miles-per-gallon.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/7-cars-miles-per-gallon.csv

--2026-03-17 04:27:32--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/7-cars-miles-per-gallon.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17321 (17K) [text/plain]
Saving to: ‘7-cars-miles-per-gallon.csv’

7-cars-miles-per-ga 100%[===================>]  16.92K  --.-KB/s    in 0.001s  

2026-03-17 04:27:33 (14.1 MB/s) - ‘7-cars-miles-per-gallon.csv’ saved [17321/17321]



In [39]:
autoData = SpContext.textFile("7-cars-miles-per-gallon.csv")
autoData.cache() # cache the RDD in memory
autoData.take(10)

['MPG,CYLINDERS,DISPLACEMENT,HORSEPOWER,WEIGHT,ACCELERATION,MODELYEAR,NAME',
 '18,8,307,130,3504,12,70,chevrolet chevelle malibu',
 '15,8,350,165,3693,11.5,70,buick skylark 320',
 '18,8,318,150,3436,11,70,plymouth satellite',
 '16,8,304,150,3433,12,70,amc rebel sst',
 '17,8,302,140,3449,10.5,70,ford torino',
 '15,8,429,198,4341,10,70,ford galaxie 500',
 '14,8,454,220,4354,9,70,chevrolet impala',
 '14,8,440,215,4312,8.5,70,plymouth fury iii',
 '14,8,455,225,4425,10,70,pontiac catalina']

In [40]:
#Remove the first line (contains headers)
dataLines = autoData.filter(lambda x: "CYLINDERS" not in x)
dataLines.count()

398

In [41]:
#Use default for average HP; broadcast it as a shared variable
avgHP =SpContext.broadcast(80.0)

In [42]:
# Function to cleanup Data
def CleanupData(inputStr) :
    global avgHP
    attList=inputStr.split(",")

    #Replace missing ? values with a normal value [data house-power column, row 128]
    hpValue = attList[3] #get the house power feature, indexed all in attList
    if hpValue == "?": #check if it is missing
        hpValue=avgHP.value #use the default value to fill in the missing

    #Create a row with cleaned up and converted data
    values= Row(
        MPG=float(attList[0]),\
        CYLINDERS=float(attList[1]), \
        DISPLACEMENT=float(attList[2]),
        HORSEPOWER=float(hpValue),\
        WEIGHT=float(attList[4]), \
        ACCELERATION=float(attList[5]), \
        MODELYEAR=float(attList[6]),\
        NAME=attList[7]
    )
    return values

In [43]:
#Run map with the CleanupData method for data cleanup, and create a Row RDD
autoMap = dataLines.map(CleanupData)
autoMap.cache()
#autoMap.take(5)
#Create a DataFrame with the rwo RDD data.
autoDf = SpSession.createDataFrame(autoMap)

print('====First 10 Row Sample')
autoDf.take(10)

====First 10 Row Sample


[Row(MPG=18.0, CYLINDERS=8.0, DISPLACEMENT=307.0, HORSEPOWER=130.0, WEIGHT=3504.0, ACCELERATION=12.0, MODELYEAR=70.0, NAME='chevrolet chevelle malibu'),
 Row(MPG=15.0, CYLINDERS=8.0, DISPLACEMENT=350.0, HORSEPOWER=165.0, WEIGHT=3693.0, ACCELERATION=11.5, MODELYEAR=70.0, NAME='buick skylark 320'),
 Row(MPG=18.0, CYLINDERS=8.0, DISPLACEMENT=318.0, HORSEPOWER=150.0, WEIGHT=3436.0, ACCELERATION=11.0, MODELYEAR=70.0, NAME='plymouth satellite'),
 Row(MPG=16.0, CYLINDERS=8.0, DISPLACEMENT=304.0, HORSEPOWER=150.0, WEIGHT=3433.0, ACCELERATION=12.0, MODELYEAR=70.0, NAME='amc rebel sst'),
 Row(MPG=17.0, CYLINDERS=8.0, DISPLACEMENT=302.0, HORSEPOWER=140.0, WEIGHT=3449.0, ACCELERATION=10.5, MODELYEAR=70.0, NAME='ford torino'),
 Row(MPG=15.0, CYLINDERS=8.0, DISPLACEMENT=429.0, HORSEPOWER=198.0, WEIGHT=4341.0, ACCELERATION=10.0, MODELYEAR=70.0, NAME='ford galaxie 500'),
 Row(MPG=14.0, CYLINDERS=8.0, DISPLACEMENT=454.0, HORSEPOWER=220.0, WEIGHT=4354.0, ACCELERATION=9.0, MODELYEAR=70.0, NAME='chevrolet

In [44]:
"""--------------------------------------------------------------------------
Perform Descriptive Data Analytics
-------------------------------------------------------------------------"""
print("=== Perform Descriptive Data Analytics ===")
#See descriptive analytics.
#use discribe to show basic statisticas for columns, such as average value, count, std, min, max
autoDf.select("MPG", "CYLINDERS", "HORSEPOWER", "WEIGHT").describe().show() #use the MPG and Cylinders as an example



#Find correlation between predictors and target (high correlation column normally have highe predictive power)
print("=== Perform Correlation Analytics ===")
for i in autoDf.columns: #loop through the columns
  if not( isinstance(autoDf.select(i).take(1)[0][0], str)) : # this is forpython 3.0+
  #if not( isinstance(autoDf.select(i).take(1)[0][0], basestring)) : #exclude the "name" coolumn which is string
    print( "Correlation to MPG for ", i, autoDf.stat.corr('MPG',i))# correlation between two columns Icorrilation between two columns i and target MPG.

=== Perform Descriptive Data Analytics ===
+-------+------------------+------------------+------------------+-----------------+
|summary|               MPG|         CYLINDERS|        HORSEPOWER|           WEIGHT|
+-------+------------------+------------------+------------------+-----------------+
|  count|               398|               398|               398|              398|
|   mean|23.514572864321615| 5.454773869346734|104.10050251256281|2970.424623115578|
| stddev| 7.815984312565783|1.7010042445332123|38.315670808035804|846.8417741973268|
|    min|               9.0|               3.0|              46.0|           1613.0|
|    max|              46.6|               8.0|             230.0|           5140.0|
+-------+------------------+------------------+------------------+-----------------+

=== Perform Correlation Analytics ===
Correlation to MPG for  MPG 1.0
Correlation to MPG for  CYLINDERS -0.7753962854205539
Correlation to MPG for  DISPLACEMENT -0.8042028248058979
Correlatio

In [45]:
"""--------------------------------------------------------------------------
Prepare data for ML
-------------------------------------------------------------------------"""
print("=== Prepare data for ML and show the 10 examples labeledPoint ===")
#Transform to a Data Frame for input to Machine Learing
#Drop columns that are not high at correlation (low correlation), use high correlation columns, for example accelleration, displacement, weight
from pyspark.ml.linalg import Vectors
def transformToLabeledPoint(row) : # tranform row to a LabelPoint object
  lp = ( row["MPG"], \
  Vectors.dense([
  row["CYLINDERS"], \
  row["DISPLACEMENT"], \
  row["HORSEPOWER"],\
  row["WEIGHT"], \
  row["ACCELERATION"], \
  row["MODELYEAR"]
  ]))
  return lp

autoLp = autoMap.map(transformToLabeledPoint) #use the method to transform to LabalPoint objects
autoDF = SpSession.createDataFrame(autoLp,["label", "features"]) # prepare the data for ML, including labels(MPG), and feature (acc, disp, weight)

autoDF.select("label","features").show(10)

=== Prepare data for ML and show the 10 examples labeledPoint ===
+-----+--------------------+
|label|            features|
+-----+--------------------+
| 18.0|[8.0,307.0,130.0,...|
| 15.0|[8.0,350.0,165.0,...|
| 18.0|[8.0,318.0,150.0,...|
| 16.0|[8.0,304.0,150.0,...|
| 17.0|[8.0,302.0,140.0,...|
| 15.0|[8.0,429.0,198.0,...|
| 14.0|[8.0,454.0,220.0,...|
| 14.0|[8.0,440.0,215.0,...|
| 14.0|[8.0,455.0,225.0,...|
| 15.0|[8.0,390.0,190.0,...|
+-----+--------------------+
only showing top 10 rows


In [46]:
"""--------------------------------------------------------------------------
Perform Machine Learning using training data
-------------------------------------------------------------------------"""
print("=== PPerform Machine Learning ===")
#Split into training and testing data
#data frame has a funtion to split the data randomly into traning and test dataset
# return two data sets: one training, one testing
(trainingData, testData) = autoDF.randomSplit([0.8, 0.2])
print("Total training data count: "+str(trainingData.count()))
print("Total testing data count: "+str(testData.count()))


#Build the model on training data
from pyspark.ml.regression import LinearRegression #we use the DataFrame machine learning package
lr = LinearRegression(maxIter=20) #iteration times
lrModel = lr.fit(trainingData) #algorithm.fit() means strat model building by training #lrModel is the trained model object
#Print the metrics
print("=== Learned model parameters ===")
print("Coefficients: " + str(lrModel.coefficients))#print out trained coefficients
print("Intercept: " + str(lrModel.intercept))#print the trained intercept
#Predict on the test data
predictions = lrModel.transform(testData) #use the model to predict the testing data
predictions.select("prediction","label","features").show()


=== PPerform Machine Learning ===
Total training data count: 331
Total testing data count: 67
=== Learned model parameters ===
Coefficients: [0.043143735119240856,0.003077904541436407,0.003479288901257951,-0.007018034648242866,0.07637488276303281,0.7712640826923806]
Intercept: -16.592424266797345
+------------------+-----+--------------------+
|        prediction|label|            features|
+------------------+-----+--------------------+
|10.676390906442727| 12.0|[8.0,350.0,160.0,...|
|12.990306189575687| 13.0|[8.0,307.0,130.0,...|
|18.585991132673477| 13.0|[8.0,318.0,150.0,...|
| 14.64171842961239| 13.0|[8.0,350.0,145.0,...|
| 9.767058413666234| 13.0|[8.0,440.0,215.0,...|
|16.620667889912802| 14.0|[8.0,304.0,150.0,...|
|12.260146021397716| 14.0|[8.0,318.0,150.0,...|
|12.911823677269265| 14.0|[8.0,351.0,153.0,...|
|12.003296169752318| 14.0|[8.0,351.0,153.0,...|
|18.541494571081692| 15.0|[8.0,318.0,150.0,...|
|16.003239798190442| 15.0|[8.0,318.0,150.0,...|
|  14.3532698566475| 15.0|[8.0

In [47]:
"""--------------------------------------------------------------------------
Perform Model Evaluation using test data
-------------------------------------------------------------------------"""
print("=== PPerform Machine Learning ===")
#Find R2 for Linear Regression to evaluate the prediciton performance: R2 value 0-1, the closer to 1 the better
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(predictionCol="prediction",
                                labelCol="label",metricName="r2") #set evaluation parameters
r2 = evaluator.evaluate(predictions)# start evaluation (compare prediction outcome labels with the real labelled outcomes)
print("The R2 (coefficient of determination) evalution result is: "+str(r2))

=== PPerform Machine Learning ===
The R2 (coefficient of determination) evalution result is: 0.8142406387102481


In [48]:
# Input your car's attributes here
my_car_name = 'Jazzy-8-2023'
my_car_cylinders = 8.0
my_car_displacement = 250.0
my_car_horsepower = 105.0
my_car_weight = 3000.0
my_car_acceleration = 17.0
my_car_modelyear = 2023.0

# Create a feature vector for your car
from pyspark.ml.linalg import Vectors
my_car_features = Vectors.dense([
    my_car_cylinders,
    my_car_displacement,
    my_car_horsepower,
    my_car_weight,
    my_car_acceleration,
    my_car_modelyear
])

# Create a DataFrame for your car's features (needs a dummy label)
myCarDf = SpSession.createDataFrame([(0.0, my_car_features)], ["label", "features"])

# Make a prediction
my_car_prediction = lrModel.transform(myCarDf)

# Display the predicted MPG
print("Predicted MPG for your car:")
my_car_prediction.select("prediction").show()

Predicted MPG for your car:
+------------------+
|        prediction|
+------------------+
|1525.3990354330767|
+------------------+

